# D3_W 팀 워크숍: 반도체 불량을 잡아라 (SECOM)

> **미션:** 불량을 최대한 잡으면서 헛경보를 줄이는 모델을 만들고, 그 근거를 설명하기

| 항목 | 내용 |
|---|---|
| 데이터 | SECOM 반도체 공정, 센서 590개, 1,567건, 불량 104건(약 6.6%) |
| 평가 데이터 | **시간 순 마지막 20%** |
| 평가 기준 | 성능 40% (Test 불량 F1) · 과정 30% (시간 분할, 누수 방지, 기준 모델 비교) · 해석 30% (지표·임계값 근거) |
| 시간 | 6교시: 1~4단계 / 7교시: 5~7단계 + 3분 공유 |

**진행 방법:** 오전 노트북(D3_01 ~ D3_05)의 코드를 복사해 고쳐 쓰세요. 막히면 기준 모델(4단계)까지만 해도 결과를 정리할 수 있습니다.

> ⚠️ SECOM은 어려운 데이터입니다. 불량 F1이 0.2~0.3만 나와도 잘한 것입니다. 점수보다 **과정과 근거**가 중요합니다.

## 0. 준비 📋

In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report, precision_recall_curve
from imblearn.pipeline import make_pipeline
from imblearn.over_sampling import SMOTE

secom = pd.read_csv("../../data/secom.csv")   # timestamp, defect(1 = 불량), s001 ~ s590

## 1. EDA: 데이터와 결측 살펴보기 ✍️

- 크기, 불량 비율
- 결측이 50% 넘는 센서는 몇 개인가?
- 값이 하나뿐인(변화 없는) 센서는 몇 개인가?

In [ ]:
# 힌트: 표.isna().mean() 은 컬럼별 결측 비율, 표.std() == 0 은 변화 없는 컬럼

## 2. 시간 기준 분할 ✍️

앞 80% Train, 마지막 20% Test. (데이터는 시간 순으로 정렬되어 있습니다)

In [ ]:
# 힌트: X.iloc[:cut], X.iloc[cut:]

## 3. 쓸 센서 고르기 (Train 기준) ✍️

어떤 센서를 뺄지는 **Train만 보고** 정합니다. 결측 50% 초과, 변화 없는 센서를 뺍니다.

In [ ]:
# 힌트: 조건을 &로 묶고, 컬럼 목록은 표.columns[조건]

## 4. 기준 모델 ✍️

① '모두 합격' 모델 ② 로지스틱 회귀. 결측 채우기·표준화는 **Pipeline 안에** 넣어야 Train 기준으로만 계산됩니다.

In [ ]:
# 힌트: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), 모델)

## 5. 불균형 대응 ✍️

클래스 가중치, SMOTE 등을 시도하고 **같은 Test로** 비교하세요. SMOTE는 결측 채우기 뒤에 넣어야 합니다.

In [ ]:
# 힌트: make_pipeline(SimpleImputer(...), StandardScaler(), SMOTE(random_state=42), 모델)

## 6. 튜닝 ✍️

교차검증은 **Train 안에서만** 합니다. 로지스틱 회귀의 규제 강도 `C`(작을수록 단순)를 바꿔 봅니다.
파이프라인 안 모델의 파라미터 이름은 `logisticregression__C`처럼 씁니다.

In [ ]:
# 힌트: GridSearchCV(파이프라인, {"logisticregression__C": [...]}, cv=..., scoring="f1")

## 7. (선택) 임계값 ✍️

Train의 마지막 20%를 Valid로 떼어 임계값별 점수를 봅니다. **Valid의 불량 건수부터 확인하세요.**
불량이 너무 적으면 임계값이 크게 흔들립니다. 그럴 때는 기본값 0.5를 유지하고 그 이유를 발표하는 것도 좋은 결론입니다.

In [ ]:
# 힌트: Valid 확률로 임계값마다 재현율·F1 비교

## 중간 점검 체크리스트

- [ ] 시간 기준으로 나눴는가? (앞 80% Train / 마지막 20% Test)
- [ ] 센서 고르기·결측 처리·스케일링 기준을 Train에서만 계산했는가?
- [ ] 리샘플링을 Train에만 적용했는가? (Pipeline 사용)
- [ ] 기준 모델과 비교했는가?
- [ ] 정확도가 아니라 F1·재현율로 보고하는가?

## 결과 공유 (팀당 3분) 💬

**1. 문제와 데이터 (30초)** 무엇을 예측했고, 어떤 점이 어려웠나?

> 답:


**2. 접근 방법 (1분)** 분할, 센서 정리, 불균형 대응, 튜닝

> 답:


**3. 결과 (1분)** 혼동행렬·F1, 고른 임계값과 이유

> 답:


**4. 배운 점 (30초)** 다음에 다르게 해 볼 것

> 답:


## 심화 선택: Steel Plates 결함 7종 다중 분류 📋

빨리 끝낸 팀의 도전 과제입니다. 정답(`fault`)이 7종류인 다중 분류입니다. 이 데이터에는 시간 정보가 없어 층화 분할을 씁니다.

In [ ]:
steel = pd.read_csv("../../data/steel_plates.csv")
print(steel["fault"].value_counts())
# 힌트: train_test_split(..., stratify=y), RandomForestClassifier, classification_report